<a href="https://colab.research.google.com/github/Gowtham12345292/Telugu-Sentiment-Analysis/blob/main/Telugu_sentimental_analysis(fine_tunning).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets

In [ ]:
import transformers
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import datasets
from datasets import load_dataset,DatasetDict
from transformers import TrainingArguments,Trainer

In [ ]:
tokenizer=AutoTokenizer.from_pretrained("google-bert/bert-base-multilingual-cased")

In [ ]:
tokenizer

In [ ]:
df=load_dataset("mounikaiiith/Telugu_Sentiment")

In [ ]:
df

In [ ]:
from datasets import DatasetDict

# Remove 'Unnamed: 0' from all splits
df = DatasetDict({
    split: dataset.remove_columns('Unnamed: 0')
    for split, dataset in df.items()
})


In [ ]:
df

In [ ]:
set(df['train']['Sentiment'])

In [ ]:
from datasets import ClassLabel

In [ ]:
cl = ClassLabel(num_classes=3, names=["neutral", "pos", "neg"])

In [ ]:
def pre_pro(batch):
  # feature variables
  l=[]
  for y in batch["Sentence"]:
      if y is not None:
         l.append(str(y))
      else:
        l.append("")
  tokenizers=tokenizer(l,max_length=512,truncation=True,padding="max_length")

  # class variables
  l1=[]
  for y1 in batch["Sentiment"]:
    if y1 is None:
      l1.append(0)
    else:
      l1.append(cl.str2int(y1))

  tokenizers["label"]=l1

  return tokenizers

In [ ]:
final_data=df.map(pre_pro,batched=True,remove_columns=["Sentence","Sentiment"])

In [ ]:
final_data

In [ ]:
model=AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-multilingual-cased",num_labels=3)

In [ ]:
model

In [ ]:
ta=TrainingArguments(eval_strategy ="epoch",
                  learning_rate=2e-5,save_strategy="epoch",
                  logging_dir="/content/models_save",
                  num_train_epochs=3,output_dir="/content/models_save",
                  per_device_train_batch_size=8,per_device_eval_batch_size=8,
                  metric_for_best_model="eval_loss",fp16=True,load_best_model_at_end=True)

In [ ]:
ta

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
tr=Trainer(model=model,
        args=ta,
        train_dataset=final_data["train"],
        eval_dataset=final_data["validation"],
        tokenizer=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])

In [ ]:
model = tr.train()

In [ ]:
# from google.colab import files
# files.download("models_save.zip")

In [ ]:
# Save the trained model, not the TrainOutput object
tr.model.save_pretrained("/content/telugu_bert_model_new")
tokenizer.save_pretrained("/content/telugu_bert_model_new")

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Load model directly from the path where you saved it
model = AutoModelForSequenceClassification.from_pretrained("/content/telugu_bert_model_new")
tokenizer = AutoTokenizer.from_pretrained("/content/telugu_bert_model_new")

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
model.push_to_hub("Gowthamvemula/Teugu_Sentimental_fine-tuning")
tokenizer.push_to_hub("Gowthamvemula/Teugu_Sentimental_fine-tuning")

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-classification", model="Gowthamvemula/Teugu_Sentimental_fine-tuning")

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("Gowthamvemula/Telugu_Sentimental_Analysis")
model = AutoModelForSequenceClassification.from_pretrained("Gowthamvemula/Telugu_Sentimental_Analysis")

In [ ]:

# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-classification", model="Gowthamvemula/Teugu_Sentimental_fine-tuning")
# Example usage:
test_text = "అన్ని టెండర్లూ పారదర్శకంగా ఉంటాయనీ, ఎల్లో మీడియా దురుద్దేశంపూర్వకంగా వార్తలు రాస్తే కచ్చితంగా శిక్షిస్తామని, పరువునష్టం దావా వేస్తామన్నారు."  # Example Telugu text
result = pipe(test_text)
result[0]

idx = int(result[0]['label'].split('_')[1])

if idx == 0:
  print("Neutral")
elif idx==1:
  print("Positive")
else:
  print("Negative")

In [ ]:
idx = int(result[0]['label'].split('_')[1])

In [ ]:
names=["neutral", "pos", "neg"]

pi

In [ ]:
# prompt: give me some 10 to 20 give me some more examples

# Example usage with different Telugu texts:
test_texts = [
    #  "ఈ సినిమా చాలా బాగుంది",  # This movie is very good
    # "నేను ఈ పుస్తకాన్ని చాలా ఇష్టపడ్డాను",  # I liked this book very much
     "ఈ ఆహారం చాలా చెడుగా ఉంది",  # This food is very bad
    "నాకు ఈ రోజు చాలా సంతోషంగా ఉంది",  # I am very happy today
     "నేను ఈ వార్తలకు చాలా బాధపడ్డాను", # I felt very sad for this news
     "ఈ వాతావరణం చాలా అద్భుతంగా ఉంది", # This weather is very wonderful
     "నేను ఈ పనిని పూర్తి చేయలేకపోతున్నాను", # I can not complete this work
     "ఈ కారు చాలా వేగంగా ఉంది",  # This car is very fast
     "ఈ పాట చాలా అందంగా ఉంది",  # This song is very beautiful
     "ఈ ప్రదేశం చాలా ప్రశాంతంగా ఉంది", # This place is very calm
    # "నేను చాలా అలసిపోయాను", # I am very tired
    #  "ఈ సమస్యకు పరిష్కారం లేదు", # There is no solution to this problem
     "నేను ఈ సమాచారాన్ని నమ్మలేకపోతున్నాను", # I can not believe this information
     "ఈ వ్యక్తి చాలా మంచివాడు", # This person is very good
     "ఈ పనిని త్వరగా పూర్తి చేయాలి" #This work should be completed quickly
    # # # Add more examples here...
]

for text in test_texts:
    result = pipe(text)
    print(f"Text: {text}")
    print(f"Result: {result}")
    print("-" * 20)
